# RegQuest ML: 04 - Daily Request Demand Forecasting
### Feature Engineering, Model Training, and Autoregressive Rollout

---
## 1. Overview
Builds leak-free autoregressive features and trains the Champion Random Forest model.


In [ ]:
import sys
from pathlib import Path
ML_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from src.regquest_ml.common.paths import PROCESSED_DATA_DIR
from src.regquest_ml.data.loader import load_processed_data
from src.regquest_ml.features.daily_demand import build_daily_features_table
from src.regquest_ml.forecasting.train import train_daily_forecaster

daily_ts = load_processed_data(str(PROCESSED_DATA_DIR / 'daily_time_series.csv'))
daily_features = build_daily_features_table(daily_ts)
print(f'Feature table size: {len(daily_features)} operating days')

forecaster = train_daily_forecaster(daily_features, model_type='random_forest')
print(f'Model trained: {forecaster.model_version} (Residual Std: {forecaster.residual_std:.2f})')


In [ ]:
from src.regquest_ml.features.calendar import generate_future_operating_dates
dates = generate_future_operating_dates('2026-11-09', number_of_days=5)
recent_history = [float(r['target_requests']) for r in daily_features[-15:]]

forecast = forecaster.autoregressive_rollout(dates, recent_history)
print('5-Day Operating Forecast Rollout:')
for item in forecast:
    print(f'  {item["date"]} ({item["weekday"]:<9}): {item["predicted_requests"]} requests (90% CI: {item["lower_bound_90"]}-{item["upper_bound_90"]})')
